기존 코드에서 발생했던 버전 충돌, 모듈 임포트 누락, API 키 환경변수 설정 오류를 모두 해결하여 최신 LangChain 버전에 맞게 새롭게 작성한 코드입니다.

주피터 노트북 환경(Google Colab 등)에서 셀(Cell) 단위로 나누어 실행할 수 있도록 정리했습니다.

### Cell 1: 패키지 설치

기존처럼 버전 충돌이 나지 않도록, 최신 패키지로 일괄 설치합니다.

In [1]:
# 최신 LangChain 생태계 패키지 및 필요한 라이브러리 일괄 설치
!pip install -qU langchain langchain-openai langchain-community langchain-chroma faiss-cpu pypdf tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.

### Cell 2: 환경변수 설정 및 모듈 임포트

누락되어 있던 `PyPDFLoader`와 `RecursiveCharacterTextSplitter` 등의 임포트 경로를 최신 버전에 맞게 추가 및 수정했습니다.

In [2]:
import os
from google.colab import userdata

# 1. OpenAI API Key 설정 (환경변수에 저장해야 LangChain이 자동으로 인식합니다)
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 2. 최신 LangChain 모듈 임포트
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# 3. 임베딩 모델 초기화
embedding = OpenAIEmbeddings()
print("OpenAI 임베딩 모델 로드 완료")

/tmp/ipykernel_2767/1546129059.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


OpenAI 임베딩 모델 로드 완료


### Cell 3: ChromaDB를 활용한 텍스트 검색 (유사도 검색 vs MMR 검색)

In [3]:
# 샘플 데이터 준비
texts = [
    "광대버섯(Amanita phalloides)은 크고 인상적인 후성(위) 자실체(담자과체)를 가지고 있습니다.",
    "큰 자실체를 가진 버섯은 Amanita phalloides입니다. 일부 품종은 모두 흰색입니다.",
    "A. phalloides, 일명 Death Cap은 알려진 모든 버섯 중에서 가장 독성이 강한 버섯 중 하나입니다."
]

# Chroma 벡터 DB 생성
smalldb = Chroma.from_texts(texts, embedding=embedding)

question = "큰 자실체를 가진 순백색 버섯에 대해 알려주세요"

# 1. 단순 유사도 검색 (Similarity Search)
print("--- 일반 유사도 검색 결과 ---")
sim_docs = smalldb.similarity_search(question, k=2)
for i, doc in enumerate(sim_docs):
    print(f"[{i+1}] {doc.page_content}")

# 2. 다양성을 고려한 검색 (Max Marginal Relevance Search, MMR)
# fetch_k로 3개를 먼저 뽑은 뒤, 가장 관련성 있으면서도 서로 다른 내용의 2개를 최종 선택합니다.
print("\n--- MMR 검색 결과 ---")
mmr_docs = smalldb.max_marginal_relevance_search(question, k=2, fetch_k=3)
for i, doc in enumerate(mmr_docs):
    print(f"[{i+1}] {doc.page_content}")

--- 일반 유사도 검색 결과 ---
[1] 큰 자실체를 가진 버섯은 Amanita phalloides입니다. 일부 품종은 모두 흰색입니다.
[2] A. phalloides, 일명 Death Cap은 알려진 모든 버섯 중에서 가장 독성이 강한 버섯 중 하나입니다.

--- MMR 검색 결과 ---
[1] 큰 자실체를 가진 버섯은 Amanita phalloides입니다. 일부 품종은 모두 흰색입니다.
[2] A. phalloides, 일명 Death Cap은 알려진 모든 버섯 중에서 가장 독성이 강한 버섯 중 하나입니다.


### Cell 4: FAISS를 활용한 PDF 문서 검색

기존 코드에서는 `page_content` 텍스트만 합쳐서 분할(`split_text`)한 뒤 다시 매핑하려다 정보 손실이 발생할 수 있는 구조였습니다. `split_documents`를 사용하여 메타데이터(페이지 번호 등)를 유지하도록 개선했습니다.

In [4]:
# 주의: 좌측 파일 탐색기에 "MachineLearning-Lecture01.pdf" 파일을 미리 업로드해야 합니다.
pdf_path = "MachineLearning-Lecture01.pdf"

if os.path.exists(pdf_path):
    # 1. PDF 로드
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    # 2. 텍스트 분할 (Chunking)
    # 텍스트만 분할하지 않고, document 자체를 분할하여 페이지 번호 메타데이터 유지
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
    splits = text_splitter.split_documents(pages)

    # 3. FAISS 벡터 DB 생성
    faiss_index = FAISS.from_documents(splits, embedding)

    # 4. 문서 검색
    query = "matlib에 대한 어떤 이야기가 있나요?"
    docs = faiss_index.similarity_search(query, k=2)

    print("--- FAISS PDF 검색 결과 ---")
    for doc in docs:
        # metadata에서 페이지 번호를 가져와 출력
        page_num = doc.metadata.get("page", "알수없음")
        print(f"[페이지 {page_num}]: {doc.page_content[:200]}...") # 너무 길면 잘리도록 200자만 출력
else:
    print(f"오류: 현재 경로에 '{pdf_path}' 파일이 없습니다. 파일을 먼저 Colab에 업로드해주세요.")

오류: 현재 경로에 'MachineLearning-Lecture01.pdf' 파일이 없습니다. 파일을 먼저 Colab에 업로드해주세요.


**수정된 주요 포인트:**

1. **버전 충돌 픽스:** `!pip install -qU langchain ...`으로 라이브러리 간 호환되는 최신 버전으로 동기화했습니다.
2. **에러 핸들링:** PDF가 없는 경우 에러를 뿜으며 멈추지 않도록 `os.path.exists`를 이용한 예외 처리를 추가했습니다.
3. **import 경로 현대화:** `langchain.vectorstores` -> `langchain_community.vectorstores` 등 LangChain 0.1.0 이상 규격에 맞추어 import 경로를 수정했습니다.